In [2]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 라이브러리 할당
import os
import cv2
import glob
from tqdm import tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# clip 영상 dir
base_dir = '/content/drive/MyDrive/OnSafe/fms_30_dataset'

# output 디렉토리
output_dir = "/content/drive/MyDrive/OnSafe/pose_csv_results"
if not os.path.exists(output_dir): # 디렉토리 없으면 생성
    os.makedirs(output_dir)

### Pose 추출

MediaPipe 이용
- 각 프레임별 keypoints 및 신뢰도 score 저장
- keypoint 좌표(x,y,z) + visibility(신뢰도) 저장

In [5]:
!pip install mediapipe opencv-python

In [6]:
# 모델 파일 (한 번만)
!wget -q -O pose_landmarker_lite.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task


In [7]:
import os
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [8]:
# pose 객체 생성
base_options = python.BaseOptions(model_asset_path="pose_landmarker_lite.task")
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

pose = vision.PoseLandmarker.create_from_options(options)

In [9]:
def extract_pose_from_video(video_paths, base_dir, detector): # 인자명을 detector로 변경
    # 비디오 파일 열기
    cap = cv2.VideoCapture(video_paths)
    if not cap.isOpened():
        print(f"[ERROR] 비디오 열 수 없음: {video_paths}")
        return None, None, None

    # 경로 및 ID 식별자 추출
    video_basename = os.path.basename(video_paths)
    file_id = os.path.splitext(video_basename)[0]

    # base_dir를 기준으로 상대 경로 계산
    rel_path = os.path.relpath(video_paths, start=base_dir)
    rel_dir = os.path.dirname(rel_path)
    video_id = rel_dir.replace(os.sep, '/') if rel_dir else "root"

    # 비디오 메타데이터 확인
    fps = cap.get(cv2.CAP_PROP_FPS)

    data = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # BGR -> RGB 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # MediaPipe Image 객체 생성
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # 포즈 추정 실행
        detection_result = detector.detect(mp_image)

        timestamp = frame_idx / fps if fps > 0 else 0

        # 결과 데이터가 있는지 확인 (비어있지 않은지)
        if detection_result.pose_landmarks:
            # [CRITICAL 수정] .landmark 가 아니라 [0] 인덱스로 접근
            for i, lm in enumerate(detection_result.pose_landmarks[0]):
                data.append({
                    "video": video_id,
                    "file_id": file_id,
                    "frame": frame_idx,
                    "timestamp": timestamp,
                    "landmark_index": i,
                    "x": lm.x,
                    "y": lm.y,
                    "z": lm.z,
                    "visibility": lm.visibility
                })

        frame_idx += 1

    cap.release()
    print(f"[INFO] Pose 추출 완료: {len(data) // 33 if data else 0} 프레임 감지됨")
    return data, video_id, file_id

## 좌표 및 기타 정보들 저장

In [10]:
def save_pose_data(data, output_dir, video_id, file_id):
    df = pd.DataFrame(data)

    # 폴더 구조 생성
    output_subdir = os.path.join(output_dir, *video_id.split('/'))
    os.makedirs(output_subdir, exist_ok=True)

    # CSV 파일 저장
    output_csv = os.path.join(output_subdir, f"{file_id}_pose.csv")
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"[INFO] 저장 완료: {output_csv} ({len(df)} 행)")

In [ ]:
# 실행부
video_paths = glob.glob(os.path.join(base_dir, 'ADL', '*.mp4'), recursive=True)
print(f"[INFO] 총 {len(video_paths)}개의 영상 처리를 시작합니다. (100개당 . 출력)")

count = 0
for video_path in video_paths:
    try:
        data, video_id, file_id = extract_pose_from_video(video_path, base_dir, pose)
        if data:
            save_pose_data(data, output_dir, video_id, file_id)

        count += 1
        if count % 100 == 0:
            print(".", end="", flush=True) # 100개마다 점 출력
        if count % 5000 == 0:
            print(f" [{count}개 완료]") # 5000개마다 줄바꿈 및 개수 표시

    except Exception as e:
        print(f"\n[ERROR] {file_id} 처리 중 실패: {e}")

print("\n[FINISH] 모든 작업이 완료되었습니다.")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
[INFO] Pose 추출 완료: 6 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/ADL/front_34_clip_05_pose.csv (198 행)
[INFO] Pose 추출 완료: 21 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/ADL/front_35_clip_00_pose.csv (693 행)
[INFO] Pose 추출 완료: 30 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/ADL/front_35_clip_01_pose.csv (990 행)
[INFO] Pose 추출 완료: 30 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/ADL/front_35_clip_02_pose.csv (990 행)
.[INFO] Pose 추출 완료: 22 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/ADL/front_35_clip_03_pose.csv (726 행)
[INFO] Pose 추출 완료: 30 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/ADL/front_36_clip_00_pose.csv (990 행)
[INFO] Pose 추출 완료: 30 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/ADL/front_36_clip_01_pose.csv (990 행)
[INFO] Pose 추출 완료: 30 프레임 감지됨
[INFO] 저장 완료: /content/drive/My

In [11]:
# 실행부
video_paths = glob.glob(os.path.join(base_dir, 'FALL', '*.mp4'), recursive=True)
print(f"[INFO] 총 {len(video_paths)}개의 영상 처리를 시작합니다. (100개당 . 출력)")

count = 0
for video_path in video_paths:
    try:
        data, video_id, file_id = extract_pose_from_video(video_path, base_dir, pose)
        if data:
            save_pose_data(data, output_dir, video_id, file_id)

        count += 1
        if count % 100 == 0:
            print(".", end="", flush=True) # 100개마다 점 출력
        if count % 5000 == 0:
            print(f" [{count}개 완료]") # 5000개마다 줄바꿈 및 개수 표시

    except Exception as e:
        print(f"\n[ERROR] {file_id} 처리 중 실패: {e}")

print("\n[FINISH] 모든 작업이 완료되었습니다.")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/FALL/C_M_09_resized_clip_01_pose.csv (990 행)
[INFO] Pose 추출 완료: 30 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/FALL/C_M_09_resized_clip_02_pose.csv (990 행)
[INFO] Pose 추출 완료: 30 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/FALL/C_M_09_resized_clip_03_pose.csv (990 행)
[INFO] Pose 추출 완료: 1 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/FALL/C_M_09_resized_clip_04_pose.csv (33 행)
[INFO] Pose 추출 완료: 29 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/FALL/20240918193500_clip_00_pose.csv (957 행)
[INFO] Pose 추출 완료: 24 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/FALL/20240918193500_clip_01_pose.csv (792 행)
[INFO] Pose 추출 완료: 30 프레임 감지됨
[INFO] 저장 완료: /content/drive/MyDrive/OnSafe/pose_csv_results/FALL/C_N_385_resized_clip_00_pose.csv (990 행)
[INFO] Pose 추출 완료: 24 프레임 감지됨
[INFO] 저장 완료:

In [12]:
fall = glob.glob(os.path.join(output_dir, 'FALL', '*.csv'))
adl = glob.glob(os.path.join(output_dir, 'ADL', '*.csv'))

print("Fall 클립 수:", len(fall))
print("ADL 클립 수:", len(adl))

Fall 클립 수: 9011
ADL 클립 수: 8578
